# Import and Helper function

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Reusable helper for the light-transform tables (driver, rider, vehicle, payments, ratings)
def to_silver_light(table_name, pk_col):
    df = spark.read.table("bronze."+table_name)

    df_silver = (
        df.dropDuplicates([pk_col])
          .withColumn("ingested_at", F.current_timestamp())
    )

    df_silver.write.mode("overwrite").saveAsTable(f"silver.{table_name}")
    print(f"silver.{table_name} written | rows={df_silver.count()}")

    return df_silver

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 3, Finished, Available, Finished, False)

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [15]:
spark.sql("SELECT count(*) FROM bronze.trips").show()

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 17, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|       4|
+--------+



# driver

In [14]:
driver_silver = to_silver_light("driver", "driver_id")

# light null/type normalization specific to driver
driver_silver = (
    driver_silver
    .withColumn("rating", F.col("rating").cast("decimal(2,1)"))
    .withColumn("gender", F.when(F.trim(F.col("gender")) == "", None).otherwise(F.col("gender")))
)

driver_silver.write.mode("overwrite").saveAsTable("silver.driver")
display(driver_silver.limit(5))

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 16, Finished, Available, Finished, False)

silver.driver written | rows=20


SynapseWidget(Synapse.DataFrame, 687e4030-676b-4114-8341-55bcae8827bf)

# rider

In [16]:
rider_silver = to_silver_light("rider", "rider_id")

rider_silver = (
    rider_silver
    .withColumn("gender", F.when(F.trim(F.col("gender")) == "", None).otherwise(F.col("gender")))
)

rider_silver.write.mode("overwrite").saveAsTable("silver.rider")
display(rider_silver.limit(5))

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 18, Finished, Available, Finished, False)

silver.rider written | rows=50


SynapseWidget(Synapse.DataFrame, 51c39de3-5f34-4154-ae1d-9411ffbf8a83)

# vehicle

In [17]:
vehicle_silver = to_silver_light("vehicle", "vehicle_id")

# data-quality check: created_at may be NULL for rows inserted before the column existed
null_created_at_count = vehicle_silver.filter(F.col("created_at").isNull()).count()
print(f"vehicle rows with NULL created_at: {null_created_at_count}")

# backfill any NULL created_at with a placeholder so downstream incremental logic doesn't break
vehicle_silver = vehicle_silver.withColumn(
    "created_at",
    F.coalesce(F.col("created_at"), F.lit("2000-01-01T00:00:00Z").cast("timestamp"))
)

vehicle_silver.write.mode("overwrite").saveAsTable("silver.vehicle")
display(vehicle_silver.limit(5))

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 19, Finished, Available, Finished, False)

silver.vehicle written | rows=20
vehicle rows with NULL created_at: 0


SynapseWidget(Synapse.DataFrame, 3a939f71-50a4-4cc4-b569-463f726835b2)

# payments

In [23]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

payments_df = spark.table("bronze.payments")

w = Window.partitionBy("trip_id").orderBy(F.col("paid_at").asc())

dedup_df = (
    payments_df
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

dedup_df.write \
    .mode("overwrite") \
    .saveAsTable("bronze.payments")

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 25, Finished, Available, Finished, False)

In [24]:
payments_silver = to_silver_light("payments", "payment_id")

payments_silver = (
    payments_silver
    .withColumn("amount", F.col("amount").cast("decimal(10,2)"))
)

payments_silver.write.mode("overwrite").saveAsTable("silver.payments")
display(payments_silver.limit(5))

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 26, Finished, Available, Finished, False)

silver.payments written | rows=4


SynapseWidget(Synapse.DataFrame, 22f00a7f-86d1-428f-ba99-bc6660249b9f)

# ratings

In [19]:
ratings_silver = to_silver_light("ratings", "rating_id")

ratings_silver = (
    ratings_silver
    .withColumn("score", F.col("score").cast("decimal(2,1)"))
)

ratings_silver.write.mode("overwrite").saveAsTable("silver.ratings")
display(ratings_silver.limit(5))

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 21, Finished, Available, Finished, False)

silver.ratings written | rows=2


SynapseWidget(Synapse.DataFrame, 11a100ff-8c5a-47bd-acda-5dd48063b642)

# trips

In [20]:
trips_bronze = spark.read.table("bronze.trips")

trips_silver = (
    trips_bronze
    .dropDuplicates(["trip_id"])
    .withColumn("ingested_at", F.current_timestamp())
)

silver_trips_terminal = trips_silver.filter(
    F.col("status").isin("completed", "cancelled")
)

silver_trips_active = trips_silver.filter(
    F.col("status").isin("requested", "confirmed", "in_progress")
)

silver_trips_terminal.write.mode("overwrite").saveAsTable("silver.trips_terminal")
silver_trips_active.write.mode("overwrite").saveAsTable("silver.trips_active")

print(f"silver.trips_terminal rows: {silver_trips_terminal.count()}")
print(f"silver.trips_active rows: {silver_trips_active.count()}")

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 22, Finished, Available, Finished, False)

silver.trips_terminal rows: 4
silver.trips_active rows: 0


# trip log table

In [22]:
log_bronze = spark.read.table("bronze.trip_event_log")

silver_trip_event_log = (
    log_bronze
    .dropDuplicates(["event_id"])
    .withColumn("ingested_at", F.current_timestamp())
)

silver_trip_event_log.write.mode("overwrite").saveAsTable("silver.trip_event_log")
print(f"silver.trip_event_log rows: {silver_trip_event_log.count()}")
display(silver_trip_event_log.orderBy("trip_id", "event_time").limit(10))

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 24, Finished, Available, Finished, False)

silver.trip_event_log rows: 16


SynapseWidget(Synapse.DataFrame, 98ed260c-d404-467d-863a-8557a78879cf)

# trip transition table

In [25]:
log = spark.read.table("silver.trip_event_log")

w = Window.partitionBy("trip_id").orderBy("event_time")

trip_transitions = (
    log
    .withColumn("prev_event_type", F.lag("event_type").over(w))
    .withColumn("prev_event_time", F.lag("event_time").over(w))
    .withColumn(
        "seconds_in_prev_state",
        F.unix_timestamp("event_time") - F.unix_timestamp("prev_event_time")
    )
    .withColumn("ingested_at", F.current_timestamp())
)

trip_transitions.write.mode("overwrite").saveAsTable("silver.trip_transitions")
display(trip_transitions.orderBy("trip_id", "event_time").limit(15))

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 09395039-0d14-4ce2-bd0b-300719732150)

In [26]:
durations = (
    trip_transitions
    .filter(F.col("prev_event_type").isNotNull())
    .withColumn(
        "duration_label",
        F.concat(F.lit("time_to_"), F.col("event_type"))
    )
    .groupBy("trip_id")
    .pivot("duration_label")
    .agg(F.first("seconds_in_prev_state"))
)

# pivot column names depend on your actual event_type values logged by the trigger
# expected something like: time_to_confirmed, time_to_in_progress, time_to_completed, time_to_cancelled
durations = durations.withColumn("ingested_at", F.current_timestamp())

durations.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("silver.trip_durations")
display(durations.limit(10))

StatementMeta(, 0966ec0e-1265-436c-98df-4433d0fdf2b9, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2f2b7d8a-1812-4a66-94b3-38efb6b598c3)